In [ ]:
def sin_fit(x, ps):
    return ps[0]*ROOT.TMath.Sin(ps[1]*x[0]+ps[2])+ps[3]

fit = ROOT.TF1("", sin_fit, , , 4)
fit.SetParameters( , , , )
h.Fit(fit)

In [ ]:
#year_hist single year 2023 sin fit
file = "UOB_2023_1.root"

f = ROOT.TFile(file)
t = f.Get("events")

f.Close()

h2023 = year_hist("2023", UoB_year_files["2023"], UNIX_year_start, ROOT.kYellow-6)

xmin = h2023.GetXaxis().GetXmin()
xmax = h2023.GetXaxis().GetXmax()

ymax = 1.1 * h2023.GetMaximum()
h2023.SetMaximum(ymax)
h2023.SetMinimum(0)

def sin_fit(x, ps):
    return ps[0]*ROOT.TMath.Sin(ps[1]*x[0]+ps[2])+ps[3]

fit = ROOT.TF1("fit2023", sin_fit, xmin, xmax, 4)

fit.SetParameters( 2e-3, 2*np.pi/(30e6), 0, 4e-3)
h2023.Fit(fit, "0")

c2023 = ROOT.TCanvas("c2023", "2023", 900, 600)
h2023.SetTitle("University of Birmingham 2023 Sine Fit")
h2023.Draw("HIST")
fit.Draw("SAME")

c2023.Draw()

In [ ]:
def hist_sin_fit(histograms, total_span, bins, omega = False):
    h_global = ROOT.TH1D("h_global_temp", "", bins, 0, total_span)
    h_global.SetDirectory(0)

    for _, h in histograms:
        h_global.Add(h)

    xmin = h_global.GetXaxis().GetXmin()
    xmax = h_global.GetXaxis().GetXmax()

    fit = ROOT.TF1("global_sin_temp", sin_fit, xmin, xmax, 4)
    fit.SetNpx(5000)

    seconds_per_year = 365.25 * 24 * 3600

    offset_guess = h_global.Integral() / h_global.GetNbinsX()
    amplitude_guess = 0.5 * offset_guess
    omega_guess = 2*np.pi/seconds_per_year

    fit.SetParameters(amplitude_guess, omega_guess, 0, offset_guess)

    if omega:
        fit.FixParameter(1, omega_guess)
    else:
        pass

    h_global.Fit(fit, "W0")

    fit.SetLineColor(ROOT.kBlack)
    fit.SetLineWidth(2)
    fit.Draw("SAME")

    return fit